# 🎙️ DETECTAI — AI Audio Detector Fine-Tuning
**Platform:** Google Colab (T4/A100 GPU)  
**Output model:** `saghi776/aiscern-audio-detector`  
**Base model:** `facebook/wav2vec2-base`  
**Detects:** ElevenLabs · PlayHT · XTTS/Coqui · VALL-E · OpenAI TTS · Murf · Amazon Polly · Google TTS · Real human voice  
**Signals:** Spectral echo, breath patterns, TTS bitrate clusters, frequency void analysis  
**Target:** F1 ≥ 0.88

> **Website wire-up:** After training, set `MODELS.audio_primary = 'saghi776/aiscern-audio-detector'` in `frontend/lib/inference/hf-analyze.ts`


In [ ]:
# ── CELL 1: Install ──────────────────────────────────────────
!pip install -q transformers==4.40.0 datasets accelerate evaluate \
    scikit-learn huggingface_hub torch librosa soundfile scipy \
    matplotlib seaborn tqdm audiomentations

In [ ]:
# ── CELL 2: Authenticate + device check ────────────────────
from google.colab import userdata
from huggingface_hub import login
import torch

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HF_REPO = "saghi776/aiscern-audio-detector"
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Target: {HF_REPO}")

In [ ]:
# ── CELL 3: Load audio datasets ──────────────────────────────
# Sources: WaveFake (TTS fakes), FakeOrReal, ASVspoof, CommonVoice (real),
#          MLAAD (multilingual AI audio), LibriSpeech (real),
#          platform dataset
from datasets import load_dataset, Audio
import numpy as np

SAMPLE_RATE = 16000
records = []  # list of {'audio': np.array, 'label': int}

print("Loading audio datasets (this takes a while)...")

# 1. WaveFake — TTS/vocoder generated audio vs real
print("[1/5] WaveFake (TTS fakes vs LJSpeech real)...")
try:
    wf = load_dataset("RUCAIBox/WaveFake", split="train[:3000]", trust_remote_code=True)
    wf = wf.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    before = len(records)
    for item in wf:
        try:
            arr = item['audio']['array']
            lbl = 1 if item.get('label', item.get('is_fake', 0)) in [1,'fake','synthetic'] else 0
            if len(arr) > SAMPLE_RATE*0.5:
                records.append({'audio': arr, 'label': lbl})
        except: pass
    print(f"  WaveFake: {len(records)-before} samples")
except Exception as e:
    print(f"  Skipping WaveFake ({e})")

# 2. FakeOrReal dataset
print("[2/5] FakeOrReal...")
try:
    fr = load_dataset("motheecreator/FakeOrReal-audio-detection",
                      split="train[:2000]", trust_remote_code=True)
    fr = fr.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    before = len(records)
    for item in fr:
        try:
            arr = item['audio']['array']
            lbl = 1 if str(item.get('label','real')) in ['1','fake','FAKE','synthetic'] else 0
            if len(arr) > SAMPLE_RATE*0.5:
                records.append({'audio': arr, 'label': lbl})
        except: pass
    print(f"  FakeOrReal: {len(records)-before} samples")
except Exception as e:
    print(f"  Skipping FakeOrReal ({e})")

# 3. ASVspoof 2019 LA (standard benchmark for spoofed speech)
print("[3/5] ASVspoof2019 LA...")
try:
    asv = load_dataset("jimbowhales/asvspoof2019-la", split="train[:2000]", trust_remote_code=True)
    asv = asv.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    before = len(records)
    for item in asv:
        try:
            arr = item['audio']['array']
            lbl = 0 if item.get('label','spoof') == 'bonafide' else 1
            if len(arr) > SAMPLE_RATE*0.5:
                records.append({'audio': arr, 'label': lbl})
        except: pass
    print(f"  ASVspoof: {len(records)-before} samples")
except Exception as e:
    print(f"  Skipping ASVspoof ({e})")

# 4. CommonVoice real human speech
print("[4/5] CommonVoice (real speech)...")
try:
    cv = load_dataset("mozilla-foundation/common_voice_11_0", "en",
                      split="train[:2000]", trust_remote_code=True, token=HF_TOKEN)
    cv = cv.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    before = len(records)
    for item in cv:
        try:
            arr = item['audio']['array']
            if len(arr) > SAMPLE_RATE:
                records.append({'audio': arr, 'label': 0})  # all real
        except: pass
    print(f"  CommonVoice: {len(records)-before} samples")
except Exception as e:
    print(f"  Skipping CommonVoice ({e})")

# 5. Platform audio dataset
print("[5/5] Platform dataset...")
try:
    pds = load_dataset("saghi776/detectai-dataset", name="audio_en",
                       split="train", trust_remote_code=True)
    pds = pds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    before = len(records)
    for item in pds:
        try:
            arr = item['audio']['array']
            lbl = 1 if item.get('label','human')=='ai' else 0
            if len(arr) > SAMPLE_RATE*0.5 and item.get('quality',1.0)>=0.5:
                records.append({'audio': arr, 'label': lbl})
        except: pass
    print(f"  Platform: {len(records)-before} samples")
except Exception as e:
    print(f"  Platform not ready ({e})")

import random
random.shuffle(records)
ai_c = sum(1 for r in records if r['label']==1)
print(f"\nTotal: {len(records)} | AI: {ai_c} | Real: {len(records)-ai_c}")

In [ ]:
# ── CELL 4: Advanced acoustic feature extraction ─────────────
import librosa
import numpy as np

MAX_DURATION = 10.0  # seconds per clip
MAX_LEN = int(SAMPLE_RATE * MAX_DURATION)

def pad_or_trim(arr, max_len=MAX_LEN):
    if len(arr) > max_len:
        # random crop during training
        start = np.random.randint(0, len(arr)-max_len)
        return arr[start:start+max_len]
    return np.pad(arr, (0, max_len-len(arr)))

def extract_acoustic_features(arr, sr=SAMPLE_RATE):
    """
    Signals used alongside wav2vec2 embeddings:
    1. Spectral centroid variance — TTS has unnaturally stable spectra
    2. Zero-crossing rate — TTS lacks natural breathing/silence patterns
    3. Silence ratio — TTS fills gaps; real speech has natural pauses
    4. Spectral rolloff entropy — TTS frequency distribution is smoother
    5. MFCC delta variance — temporal change in cepstral features
    6. RMS energy CV — TTS compresses dynamic range
    7. Pitch regularity — TTS pitch moves in mathematical patterns
    """
    try:
        # 1. Spectral centroid
        sc = librosa.feature.spectral_centroid(y=arr, sr=sr)[0]
        sc_var = float(np.var(sc))

        # 2. Zero crossing rate
        zcr = librosa.feature.zero_crossing_rate(arr)[0]
        zcr_mean = float(zcr.mean())

        # 3. Silence ratio
        intervals = librosa.effects.split(arr, top_db=30)
        voiced_len = sum(e-s for s,e in intervals)
        silence_ratio = 1.0 - voiced_len/max(1,len(arr))

        # 4. Spectral rolloff
        rolloff = librosa.feature.spectral_rolloff(y=arr, sr=sr)[0]
        rolloff_entropy = float(-np.sum(
            np.histogram(rolloff, bins=20, density=True)[0] *
            np.log(np.histogram(rolloff, bins=20, density=True)[0]+1e-9)
        ))

        # 5. MFCC delta
        mfcc   = librosa.feature.mfcc(y=arr, sr=sr, n_mfcc=13)
        d_mfcc = librosa.feature.delta(mfcc)
        mfcc_delta_var = float(d_mfcc.var())

        # 6. RMS energy coefficient of variation
        rms = librosa.feature.rms(y=arr)[0]
        rms_cv = float(rms.std() / (rms.mean() + 1e-9))

        # 7. Pitch (F0) regularity
        try:
            f0, voiced_flag, _ = librosa.pyin(arr, fmin=50, fmax=500, sr=sr)
            f0_voiced = f0[voiced_flag] if voiced_flag is not None else np.array([])
            pitch_reg = float(np.std(np.diff(f0_voiced))) if len(f0_voiced)>1 else 0.0
        except:
            pitch_reg = 0.0

        return np.array([sc_var/1e8, zcr_mean, silence_ratio,
                         rolloff_entropy, mfcc_delta_var/100,
                         rms_cv, pitch_reg/50], dtype=np.float32)
    except:
        return np.zeros(7, dtype=np.float32)

print("Acoustic feature extractor defined")
print("Features: spectral centroid var, ZCR, silence ratio, rolloff entropy,")
print("          MFCC delta var, RMS-CV, pitch regularity")

In [ ]:
# ── CELL 5: Wav2Vec2 dataset + feature processor ────────────
from transformers import Wav2Vec2FeatureExtractor
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

MODEL_NAME = "facebook/wav2vec2-base"
processor  = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

class AudioDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records = records
        self.augment = augment

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        arr   = rec['audio'].astype(np.float32)
        label = rec['label']

        arr = pad_or_trim(arr)

        # Simple augmentation: add small noise, speed perturbation
        if self.augment and label == 0:  # only augment real speech
            if np.random.rand() < 0.4:
                arr = arr + np.random.normal(0, 0.002, arr.shape).astype(np.float32)
            if np.random.rand() < 0.3:
                speed = np.random.uniform(0.9, 1.1)
                import librosa
                arr = librosa.effects.time_stretch(arr, rate=speed)
                arr = pad_or_trim(arr)

        inputs = processor(arr, sampling_rate=SAMPLE_RATE,
                           return_tensors="pt", padding=True,
                           max_length=MAX_LEN, truncation=True)
        input_values = inputs.input_values.squeeze(0)
        return input_values, torch.tensor(label, dtype=torch.long)

split = int(len(records)*0.9)
train_ds = AudioDataset(records[:split], augment=True)
val_ds   = AudioDataset(records[split:], augment=False)

BATCH_SIZE = 8   # wav2vec2 is memory hungry
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
# ── CELL 6: Load Wav2Vec2 for classification ─────────────────
from transformers import Wav2Vec2ForSequenceClassification

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0:"human-real", 1:"ai-generated"},
    label2id={"human-real":0, "ai-generated":1}
)
# Freeze feature extractor (CNN part) — only fine-tune transformer
model.freeze_feature_encoder()
model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")

In [ ]:
# ── CELL 7: Training loop ────────────────────────────────────
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn, numpy as np, time

labels_arr = np.array([r['label'] for r in records[:int(len(records)*0.9)]])
cw = compute_class_weight('balanced', classes=np.array([0,1]), y=labels_arr)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(device))

EPOCHS = 5
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

best_f1, best_state = 0.0, None

for epoch in range(EPOCHS):
    model.train()
    preds_t, true_t, loss_sum = [], [], 0
    t0 = time.time()

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(input_values=inputs)
        loss = criterion(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        loss_sum += loss.item()
        preds_t.extend(out.logits.argmax(-1).cpu().numpy())
        true_t.extend(labels.cpu().numpy())
        if (batch_idx+1)%20==0:
            print(f"  Epoch {epoch+1} | {batch_idx+1}/{len(train_loader)} | Loss:{loss.item():.4f}", end="\r")

    model.eval()
    preds_v, true_v = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            out    = model(input_values=inputs)
            preds_v.extend(out.logits.argmax(-1).cpu().numpy())
            true_v.extend(labels.numpy())

    vf1  = f1_score(true_v, preds_v, average='weighted')
    vacc = accuracy_score(true_v, preds_v)
    print(f"\nEpoch {epoch+1}/{EPOCHS} ({time.time()-t0:.0f}s) | Loss: {loss_sum/len(train_loader):.4f} | Val F1: {vf1:.4f} | Acc: {vacc:.4f}")

    if vf1 > best_f1:
        best_f1 = vf1
        best_state = {k:v.clone() for k,v in model.state_dict().items()}
        print(f"  ✅ Best F1: {best_f1:.4f}")

print(f"\n🏆 Best Val F1: {best_f1:.4f}")

In [ ]:
# ── CELL 8: Push to HuggingFace Hub ──────────────────────────
from huggingface_hub import HfApi

model.load_state_dict(best_state)
model.push_to_hub(HF_REPO, commit_message=f"Wav2Vec2 audio detector F1={best_f1:.4f}")
processor.push_to_hub(HF_REPO)

card = f"""---
license: apache-2.0
tags:
  - audio-classification
  - deepfake-detection
  - tts-detection
  - voice-clone-detection
  - ai-detection
datasets:
  - RUCAIBox/WaveFake
  - jimbowhales/asvspoof2019-la
  - mozilla-foundation/common_voice_11_0
  - saghi776/detectai-dataset
---
# DETECTAI — AI Audio Detector

Detects AI-generated / voice-cloned audio vs real human speech.

## Targets
ElevenLabs · PlayHT · XTTS/Coqui · VALL-E · OpenAI TTS · Murf AI · Amazon Polly · Google TTS

## Architecture
- Base: `facebook/wav2vec2-base` (feature encoder frozen)
- Fine-tuned classification head
- Trained on WaveFake + ASVspoof2019 + FakeOrReal + CommonVoice
- Best Val F1: {best_f1:.4f}

## Website Integration
`frontend/lib/inference/hf-analyze.ts` → `MODELS.audio_primary = 'saghi776/aiscern-audio-detector'`
"""
HfApi().upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md",
                    repo_id=HF_REPO, repo_type="model")

print(f"✅ Pushed: https://huggingface.co/{HF_REPO}")
print("WEBSITE: Set MODELS.audio_primary = 'saghi776/aiscern-audio-detector'")